# Notebook 27 — Q4 blind failure evaluation

The Q4 2021 ZIP must be available. The first code cell checks the usual `data/raw/data_Q4_2021.zip` location, then searches the component `data` directory. Set `Q4_ZIP_OVERRIDE` to the full archive path if it is elsewhere.

Run only after Notebook 26 has written `q4_scoring_complete.json` and `q4_score_lock.json`. The first cell verifies every frozen input and every daily score partition **before** the first Q4 failure label is read. The event census includes reserved failed drives even when no 30-day score exists. Scores and thresholds remain fixed. The per-model rows are the primary analysis; the pooled rows are descriptive. Undefined metrics are left blank. No winner is selected from Q4, and this is not an operational safety approval.

In [4]:
from pathlib import Path
from datetime import datetime,timezone
from sklearn.metrics import average_precision_score
import hashlib,json,re,zipfile
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
ROOT=COMPONENT_ROOT/'results'/RUN_ID
LOCK=ROOT/'q4_blind_candidate_protocol_v1'
SCORES=ROOT/'q4_feature_only_scoring_v1'
OUT=ROOT/'q4_blind_failure_evaluation_v1'
OUT.mkdir(parents=True,exist_ok=True)
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(4*1024*1024),b''):h.update(block)
    return h.hexdigest()
def read_json(path):
    assert path.is_file(),f'Missing prerequisite: {path}'
    return json.loads(path.read_text())
def freeze_json(path,obj):
    if path.exists():assert read_json(path)==obj,f'Frozen artifact changed: {path}'
    else:path.write_text(json.dumps(obj,indent=2))
lock_path=LOCK/'q4_candidate_lock.json';lock=read_json(lock_path);protocol=lock['protocol']
assert sha256(lock_path)==read_json(LOCK/'q4_protocol_lock_complete.json')['candidate_lock_sha256']
assert sha256(LOCK/'q4_blind_protocol.json')==read_json(LOCK/'q4_protocol_lock_complete.json')['protocol_sha256']
for key,item in lock['artifacts'].items():
    assert sha256(COMPONENT_ROOT/item['path'])==item['sha256'],key
score_lock_path=SCORES/'q4_score_lock.json';score_lock=read_json(score_lock_path)
score_complete=read_json(SCORES/'q4_scoring_complete.json')
manifest_path=SCORES/'q4_score_manifest.json';manifest=read_json(manifest_path)
assert score_complete['analysis_complete'] and sha256(score_lock_path)==score_complete['score_lock_sha256']
assert sha256(manifest_path)==score_lock['score_manifest_sha256']==score_complete['score_manifest_sha256']
assert score_lock['candidate_lock_sha256']==manifest['candidate_lock_sha256']==sha256(lock_path)
Q4_ZIP_OVERRIDE = None  # Example: Path(r'D:/datasets/data_Q4_2021.zip')
# If unset, find one Q4 2021 archive inside the component's data folder.
if Q4_ZIP_OVERRIDE is not None:
    Q4_ZIP = Path(Q4_ZIP_OVERRIDE).expanduser()
    if not Q4_ZIP.is_file():
        raise FileNotFoundError(f'Q4_ZIP_OVERRIDE points to a missing file: {Q4_ZIP}')
else:
    expected_q4 = COMPONENT_ROOT / 'data/raw/data_Q4_2021.zip'
    if expected_q4.is_file():
        Q4_ZIP = expected_q4
    else:
        data_dir = COMPONENT_ROOT / 'data'
        candidates = sorted(p for p in data_dir.rglob('*.zip')
                            if 'q4' in p.name.lower() and '2021' in p.name.lower()) if data_dir.is_dir() else []
        if len(candidates) == 1:
            Q4_ZIP = candidates[0]
            print('Using Q4 archive:', Q4_ZIP)
        else:
            raise FileNotFoundError(
                f'Q4 2021 ZIP not found at {expected_q4}. '
                f'Found {len(candidates)} Q4 2021 ZIP candidates under {data_dir}: {candidates}. '
                'Place data_Q4_2021.zip at the expected path, or set Q4_ZIP_OVERRIDE '
                'above to the exact full path of your existing archive. '
                'Notebook 25 did not check whether this archive was present.')
assert score_lock['q4_archive_sha256']==manifest['q4_archive_sha256']==sha256(Q4_ZIP)
assert not score_lock['failure_column_read'] and not score_lock['q4_failure_outcomes_opened']
assert not manifest['failure_column_read'] and not manifest['q4_failure_outcomes_opened']
assert score_complete['score_rows']==score_lock['score_rows']==manifest['score_rows']
assert protocol['primary_method']=='transformer_sector_error' and not protocol['winner_selection_after_q4']
DATES=pd.date_range('2021-10-01','2021-12-31').strftime('%Y-%m-%d').tolist()
DAY={d:i for i,d in enumerate(DATES)}
assert len(DATES)==len(manifest['partition_sha256'])==92
for date in DATES:
    path=SCORES/'daily_scores'/f'{date}.parquet'
    assert sha256(path)==manifest['partition_sha256'][path.name]
    day_meta=read_json(path.with_suffix('.json'))
    assert day_meta['sha256']==sha256(path) and day_meta['stats']==manifest['daily_quality'][date]
models=list(lock['future_test_drives_by_model'])
reservation=read_json(COMPONENT_ROOT/lock['artifacts']['reservation']['path'])
future={str(s):m for m,g in reservation['models'].items() for s in g['future_test']}
assert {m:sum(x==m for x in future.values()) for m in models}==lock['future_test_drives_by_model']
methods={'transformer_sector_error':('sector_error','transformer_sector_error'),
         'transformer_temp_error':('temp_error','transformer_temp_error'),
         'transformer_joint_error':('joint_error','transformer_joint_error'),
         'isolation_forest':('isolation_forest','isolation_forest_flag'),
         'sector_positive_fixed':('sector_positive','sector_positive_fixed'),
         'sector_delta_fixed':('sector_delta','sector_delta_fixed'),
         'sector_positive_calibrated':('sector_positive','sector_positive_calibrated'),
         'sector_delta_calibrated':('sector_delta','sector_delta_calibrated')}
assert list(methods)==[protocol['primary_method']]+protocol['secondary_methods']+protocol['comparators']
assert len(methods)==lock['frozen_method_count']==8
horizons=[protocol['primary_horizon_days']]+protocol['other_horizons_days']
assert horizons==[14,7,30]
settings={'candidate_lock_sha256':sha256(lock_path),'score_lock_sha256':sha256(score_lock_path),
          'q4_archive_sha256':sha256(Q4_ZIP),'methods':list(methods),
          'horizons':horizons,'winner_selection_after_q4':False}
freeze_json(OUT/'settings.json',settings)
print('Blind protocol, all artifacts and 92 scored partitions verified. Labels still unopened.')

Blind protocol, all artifacts and 92 scored partitions verified. Labels still unopened.


## One labeled Q4 census after score lock

In [5]:
with zipfile.ZipFile(Q4_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
    assert set(members)==set(DAY)
    observed={};first={};failure_rows=[];all_seen=0
    # First and only labeled Q4 pass. Read only ID, model, date and failure.
    for date in DATES:
        seen=set();d=DAY[date]
        with z.open(members[date]) as f:
            for chunk in pd.read_csv(f,usecols=['date','serial_number','model','failure'],
                                     dtype={'serial_number':str},chunksize=50000):
                part=chunk.loc[chunk.serial_number.isin(future)]
                if part.empty:continue
                assert part.date.astype(str).eq(date).all()
                ss=part.serial_number.tolist()
                assert len(ss)==len(set(ss)) and seen.isdisjoint(ss),'Duplicate drive/day'
                seen.update(ss);all_seen+=len(ss)
                assert all(future[s]==m for s,m in part[['serial_number','model']].itertuples(index=False,name=None))
                flags=pd.to_numeric(part.failure,errors='raise').to_numpy()
                assert np.isin(flags,[0,1]).all(),'Failure must be a 0/1 indicator'
                for s,flag in zip(ss,flags):
                    observed[s]=observed.get(s,0)|(1<<d)
                    if flag==1:
                        failure_rows.append((s,future[s],date))
                        first.setdefault(s,d)
        if date.endswith(('-01','-15')):print('Q4 label census:',date,'observed:',len(seen))
recorded=pd.DataFrame(failure_rows,columns=['serial_number','model','failure_date'])
assert not recorded.duplicated(['serial_number','failure_date']).any()
events=pd.DataFrame([(s,future[s],DATES[d]) for s,d in first.items()],
                    columns=['serial_number','model','failure_date'])
events=events.sort_values(['model','failure_date','serial_number']).reset_index(drop=True)
assert set(events.serial_number)<=set(future)
# A failure drive can be missing from score files; it stays in this census.
print('Q4 reserved observed drive-days:',all_seen,'| first recorded failure drives:',len(events))


Q4 label census: 2021-10-01 observed: 19292
Q4 label census: 2021-10-15 observed: 19279
Q4 label census: 2021-11-01 observed: 19255
Q4 label census: 2021-11-15 observed: 19248
Q4 label census: 2021-12-01 observed: 19254
Q4 label census: 2021-12-15 observed: 19253
Q4 reserved observed drive-days: 1771961 | first recorded failure drives: 51


## Complete-history events and censored endpoint metrics

In [6]:
KEYS=['serial_number','model','end_date']
frames=[]
for date in DATES:
    part=pd.read_parquet(SCORES/'daily_scores'/f'{date}.parquet')
    assert len(part)==manifest['daily_quality'][date]['scored_rows']
    assert part.end_date.eq(date).all() and not part.duplicated(KEYS).any()
    assert all(future[s]==m for s,m in part[['serial_number','model']].itertuples(index=False,name=None))
    for method,(score_col,flag_col) in methods.items():
        assert np.isfinite(part[score_col].to_numpy(dtype=float)).all()
        assert not part[flag_col].isna().any()
        expected=part[score_col].to_numpy(dtype=float)
        if method.startswith('transformer_'):
            thresholds=np.array([lock['thresholds']['transformer'][m][score_col] for m in part.model])
        elif method=='isolation_forest':
            thresholds=np.array([lock['thresholds']['isolation_forest'][m] for m in part.model])
        elif method.endswith('_fixed'):
            thresholds=0
        else:
            thresholds=np.array([lock['thresholds']['smart'][m][score_col] for m in part.model])
        np.testing.assert_array_equal(part[flag_col].to_numpy(dtype=bool),expected>thresholds)
    frames.append(part)
scored=pd.concat(frames,ignore_index=True)
assert len(scored)==manifest['score_rows'] and not scored.duplicated(KEYS).any()
assert len(scored)>0
scored['day']=scored.end_date.map(DAY).astype(int)
# Index each drive once; event history exists only with ALL h consecutive pre-failure endpoints.
by_drive={s:g.set_index('day') for s,g in scored.groupby('serial_number',sort=False)}
ALL_MASK=(1<<len(DATES))-1
def has_all(bits,lo,hi):
    # NumPy day indices overflow when shifted beyond 63 bits. Use Python ints.
    bits=int(bits);lo=int(lo);hi=int(hi)
    if lo<0 or hi>=len(DATES):return False
    mask=((1<<(hi-lo+1))-1)<<lo
    return (bits&mask)==mask

def calculate(group_name,model,method,h):
    score_col,flag_col=methods[method]
    g=scored if model is None else scored.loc[scored.model.eq(model)]
    scores=g[score_col].to_numpy(dtype=float)
    flags=g[flag_col].to_numpy(dtype=bool)
    serials=g.serial_number.to_numpy();days=g.day.to_numpy(dtype=int)
    target=np.full(len(g),-1,dtype=np.int8)
    for i,(s,d) in enumerate(zip(serials,days)):
        d=int(d)
        f=first.get(s)
        if f is not None and f<=d:continue
        if f is not None and f-d<=h:target[i]=1
        elif has_all(observed.get(s,0),d+1,d+h):target[i]=0
    keep=target>=0;y=target[keep];s=scores[keep];a=flags[keep]
    pos=int((y==1).sum());neg=int((y==0).sum())
    tp=int(np.sum(a&(y==1)));fp=int(np.sum(a&(y==0)))
    this_events=events if model is None else events.loc[events.model.eq(model)]
    details=[]
    for event in this_events.itertuples(index=False):
        d=DAY[event.failure_date];required=range(d-h,d)
        history=by_drive.get(event.serial_number)
        complete=bool(d-h>=0 and history is not None and all(i in history.index for i in required))
        alarms=history.loc[list(required),flag_col].to_numpy(dtype=bool) if complete else np.array([],dtype=bool)
        fired=np.flatnonzero(alarms)
        lead=int(h-fired[0]) if len(fired) else np.nan
        details.append({'serial_number':event.serial_number,'model':event.model,'group':group_name,
                        'method':method,'horizon_days':h,'failure_date':event.failure_date,
                        'complete_warning_history':complete,'recalled':bool(len(fired)),
                        'first_warning_lead_days':lead})
    detail=pd.DataFrame(details)
    ncomplete=int(detail.complete_warning_history.sum()) if len(detail) else 0
    recalled=int(detail.recalled.sum()) if len(detail) else 0
    leads=detail.loc[detail.recalled,'first_warning_lead_days'] if len(detail) else []
    row={'group':group_name,'method':method,'horizon_days':h,'scored_endpoints':len(g),
         'evaluable_endpoints':int(keep.sum()),'positive_endpoints':pos,'negative_endpoints':neg,
         'censored_or_post_failure_endpoints':int((~keep).sum()),
         'positive_prevalence':float(pos/(pos+neg)) if pos+neg else np.nan,
         'average_precision':float(average_precision_score(y,s)) if pos and neg else np.nan,
         'threshold_precision':tp/(tp+fp) if tp+fp else np.nan,
         'positive_endpoint_recall':tp/pos if pos else np.nan,
         'false_alerts_per_1000_observed_negative_endpoints':1000*fp/neg if neg else np.nan,
         'all_scored_alerts_per_1000_endpoints':1000*float(flags.mean()) if len(g) else np.nan,
         'recorded_first_failure_events':len(this_events),'events_with_complete_warning_history':ncomplete,
         'events_incomplete_or_unscoreable':len(this_events)-ncomplete,
         'events_recalled':recalled,'strict_event_recall':recalled/ncomplete if ncomplete else np.nan,
         'median_first_warning_lead_days':float(np.median(leads)) if len(leads) else np.nan,
         'status':'both_endpoint_classes' if pos and neg else 'insufficient_endpoint_classes'}
    return row,detail
rows=[];details=[]
for name,model in [(m,m) for m in models]+[('pooled_descriptive',None)]:
    for method in methods:
        for h in horizons:
            row,detail=calculate(name,model,method,h)
            rows.append(row);details.append(detail)
metrics=pd.DataFrame(rows)
event_details=pd.concat(details,ignore_index=True) if len(events) else pd.DataFrame(
    columns=['serial_number','model','group','method','horizon_days','failure_date',
             'complete_warning_history','recalled','first_warning_lead_days'])
assert len(metrics)==(len(models)+1)*len(methods)*len(horizons)
assert int(metrics.loc[metrics.group.eq('pooled_descriptive') & metrics.horizon_days.eq(14),
                       'recorded_first_failure_events'].min())==len(events)
print('Primary 14-day per-model outcome (undefined metrics mean insufficient evidence):')
display(metrics.loc[metrics.method.eq(protocol['primary_method'])&metrics.horizon_days.eq(14),
    ['group','positive_endpoints','negative_endpoints','average_precision','recorded_first_failure_events',
     'events_with_complete_warning_history','strict_event_recall','false_alerts_per_1000_observed_negative_endpoints']])

Primary 14-day per-model outcome (undefined metrics mean insufficient evidence):


,group,positive_endpoints,negative_endpoints,average_precision,recorded_first_failure_events,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints
0,TOSHIBA MG07ACA14TA,161,496100,0.020002,14,9,0.333333,2.439024
24,ST12000NM0008,100,289806,0.025943,8,6,0.333333,2.867435
48,ST4000DM000,236,288881,0.060248,19,15,0.333333,1.509272
72,ST8000NM0055,59,220535,0.007758,8,2,0.500000,2.081302
96,HGST HMS5C4040BLE640,28,193320,0.001073,2,2,0.000000,0.615560
120,pooled_descriptive,584,1488642,0.020189,51,34,0.323529,2.052206


## Conditional drive-level uncertainty for primary event recall

In [7]:
# A paired drive resample is meaningful only with ten distinct complete event drives.
# This interval concerns event recall, not AP or deployment safety.
rng=np.random.default_rng(42)
ci_rows=[]
for name in models+['pooled_descriptive']:
    sub=event_details.loc[event_details.group.eq(name)&
        event_details.method.eq(protocol['primary_method'])&
        event_details.horizon_days.eq(14)&event_details.complete_warning_history].copy()
    if sub.serial_number.nunique()<10:
        ci_rows.append({'group':name,'complete_event_drives':sub.serial_number.nunique(),
                        'bootstrap_replicates':0,'strict_event_recall_ci_low':np.nan,
                        'strict_event_recall_ci_high':np.nan})
        continue
    by_serial=sub.groupby('serial_number').recalled.agg(['sum','count'])
    arr=by_serial[['sum','count']].to_numpy(dtype=float)
    draw=rng.integers(0,len(arr),size=(1000,len(arr)))
    sampled=arr[draw]
    estimates=sampled[:,:,0].sum(axis=1)/sampled[:,:,1].sum(axis=1)
    low,high=np.quantile(estimates,[.025,.975])
    ci_rows.append({'group':name,'complete_event_drives':len(arr),'bootstrap_replicates':1000,
                    'strict_event_recall_ci_low':float(low),'strict_event_recall_ci_high':float(high)})
uncertainty=pd.DataFrame(ci_rows)
print('Primary 14-day drive-level event recall uncertainty:')
display(uncertainty)


Primary 14-day drive-level event recall uncertainty:


,group,complete_event_drives,bootstrap_replicates,strict_event_recall_ci_low,strict_event_recall_ci_high
0,TOSHIBA MG07ACA14TA,9,0,NaN,NaN
1,ST12000NM0008,6,0,NaN,NaN
2,ST4000DM000,15,1000,0.133333,0.600000
3,ST8000NM0055,2,0,NaN,NaN
4,HGST HMS5C4040BLE640,2,0,NaN,NaN
5,pooled_descriptive,34,1000,0.176471,0.470588


## Record the blind outcome and provenance

In [8]:
def frozen_csv(path,frame):
    if path.exists():
        existing=pd.read_csv(path)
        assert len(existing)==len(frame),f'Existing result row count changed: {path}'
        # Recompute bytes in memory for deterministic replay without altering a published artifact.
        assert path.read_bytes()==frame.to_csv(index=False).encode('utf-8'),f'Existing result changed: {path}'
    else:frame.to_csv(path,index=False)
frozen_csv(OUT/'q4_primary_event_recall_uncertainty.csv',uncertainty)
frozen_csv(OUT/'q4_first_failure_census.csv',events)
frozen_csv(OUT/'q4_blind_failure_metrics.csv',metrics)
frozen_csv(OUT/'q4_blind_event_details.csv',event_details)
coverage=pd.DataFrame([{'date':d,**manifest['daily_quality'][d]} for d in DATES])
# Daily quality is kept in the immutable score manifest; the summary only aggregates it.
summary={'first_failure_drives':len(events),'failure_rows':len(recorded),
         'reserved_drives':len(future),'observed_drive_days':all_seen,
         'scored_endpoints':len(scored),
         'observed_drives':len(observed),
         'unscored_failure_drives':int(sum(s not in by_drive for s in first)),
         'daily_quality_totals':{field:int(sum(manifest['daily_quality'][d][field] for d in DATES))
             for field in ['observed_rows','invalid_rows','gap_resets','scored_rows']},
         'primary_method':protocol['primary_method'],'primary_horizon_days':14,
         'score_lock_sha256':sha256(score_lock_path),'winner_selection_after_q4':False,
         'uncertainty':'Drive-level primary-event recall bootstrap reported only for groups with at least ten complete event drives; other intervals are undefined.'}
freeze_json(OUT/'q4_evaluation_summary.json',summary)
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'candidate_lock_sha256':sha256(lock_path),'score_lock_sha256':sha256(score_lock_path),
        'settings_sha256':sha256(OUT/'settings.json'),
        'census_sha256':sha256(OUT/'q4_first_failure_census.csv'),
        'uncertainty_sha256':sha256(OUT/'q4_primary_event_recall_uncertainty.csv'),
        'metrics_sha256':sha256(OUT/'q4_blind_failure_metrics.csv'),
        'event_details_sha256':sha256(OUT/'q4_blind_event_details.csv'),
        'summary_sha256':sha256(OUT/'q4_evaluation_summary.json'),
        'recorded_first_failure_drives':len(events),'winner_selection_after_q4':False,
        'sealed_June_test_opened':False}
completion_path=OUT/'q4_blind_evaluation_complete.json'
if completion_path.exists():
    old=read_json(completion_path)
    assert {k:v for k,v in old.items() if k!='created_utc'}=={k:v for k,v in record.items() if k!='created_utc'}
else:completion_path.write_text(json.dumps(record,indent=2))
print('NOTEBOOK 27 COMPLETE — Q4 blind evaluation recorded. Candidate selection remains frozen.')
print('Next: review primary method per model, censoring and event counts before operational integration.')

NOTEBOOK 27 COMPLETE — Q4 blind evaluation recorded. Candidate selection remains frozen.
Next: review primary method per model, censoring and event counts before operational integration.
